# 用 Sciverse 做专利与文献交叉探索

> 通过专利元数据检索与论文语义检索探索专利和学术文献，发现技术关联

**场景**: 研发人员需要了解某项技术在专利和学术论文中的覆盖情况，分别检索两个集合，分析关联和差异。

**使用接口**: meta-search（search_metadata）、agentic-search（semantic_search）、content（read_content）

**预估调用量**: ~10–20 次 API 调用

---


## Step 1: 环境准备

安装依赖并配置环境变量


In [ ]:
!pip install httpx anthropic
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # 替换为你的真实值


## Step 2: 检索专利元数据和论文片段

专利用 collection=patents 检索元数据；semantic_search 仍只用于论文。


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search(query: str, top_k: int = 15):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        resp.raise_for_status()
        return (resp.json().get("hits") or [])

async def search_patents(query: str):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS,
            json={"collection": "patents", "query": query, "page_size": 15},
        )
        resp.raise_for_status()
        return resp.json().get("results") or []

async def read_patent(doc_id: str):
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content", headers=HEADERS,
            params={"collection": "patents", "doc_id": doc_id, "offset": 0, "limit": 4096},
        )
        resp.raise_for_status()
        return resp.json()

async def main():
    patent_hits = await search_patents("CRISPR base editing")
    academic_hits = await search("CRISPR base editing adenine cytosine mechanism")
    print(f"Patents: {len(patent_hits)} metadata records; papers: {len(academic_hits)} chunks")
    return patent_hits, academic_hits

patent_hits, academic_hits = await main()


## Step 3: 交叉分析与报告生成

将两组检索结果交给 LLM 进行关联分析


In [ ]:
from anthropic import Anthropic

client = Anthropic()

patent_summary = "\n".join([
    f"- [{h.get('doc_id', 'no doc_id')}] {h.get('title', '')}: {h.get('abstract', '')}"
    for h in patent_hits[:8]
])
academic_summary = "\
".join([
    f"- [{h['doc_id']}] (score: {h['score']:.2f}) {h['title']}: {h.get('chunk', '')[:80]}..."
    for h in academic_hits[:8]
])

msg = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": f"""分析以下两组检索结果的技术关联：

## 专利元数据（不能视为已验证全文）
{patent_summary}

## 学术文献片段
{academic_summary}

请输出：
1) 两组结果中的技术主题对比
2) 可能的专利-论文关联（基于内容相似性）
3) 技术发展脉络推测

注意：所有结论必须基于上述检索结果，标注 doc_id。"""
    }]
)
print(msg.content[0].text)


## 注意事项

- 专利通过 search_metadata(collection="patents") 获取；semantic_search 只返回论文。
- 过滤或排序前调用 list_catalog(collection="patents") 检查字段。
- 专利元数据使用响应的 results 容器；不假设存在论文语义片段的 score/chunk。
- 关键专利若返回可读 doc_id，可调用 read_patent(doc_id) 验证原文；后续读取沿用 collection="patents" 和 next_offset。
- get_resource 主要获取论文图片，不新增 ZIP 能力。


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
